In [26]:
# library imports
import pandas as pd                             # tables, dataframes
import numpy as np                              # arrays, numerical computations
from uncertainties import ufloat as uf          # uncertainty propagation

from pathlib import Path                        # paths in operating system
# path config
downloads_dir = Path.home() / 'Downloads'
# make sure the folder exists (just in case)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'my_plot.pdf'

In [27]:
# custom functions

# type A uncertainty
def unc_A(data):
    '''
    Type A uncertainty (standard error of the mean)

    data : list or numpy array
        measurements
    return : float
        type A uncertainty
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)

    return u_A

# loading from df into numpy, needs .T for the correct allocation
def df_to_numpy_matrix(df, column, dtype=float):
    '''
    Converts the selected columns of a dataframe to a numpy array

    df : dataframe
        dataframe to be converted to a numpy object
    column : str or list[str]
        column name (str) or list of column names (list[str])
    dtype : type
        data type of the resulting array (default float)
    return : numpy.ndarray
        matrix of the selected columns; 1D array if a single column is given
    '''
    if isinstance(column, str):
        column = [column]

    missing = [s for s in column if s not in df.columns]
    if missing:
        raise KeyError(f'Missing columns: {missing}')

    data = df[column].to_numpy(dtype=dtype)

    return data[:, 0] if len(column) == 1 else data

# nacteni z df do jednotlivych arrays
def df_to_arrays(df, sloupce, od=0, do=None, dtype=float):
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    return [df[s].iloc[od:do].dropna().to_numpy(dtype=dtype) for s in sloupce]

# ufloat z aritmetickeho prumeru
def ufloat_unc_A(array):
    return uf(np.mean(array), unc_A(array))

# ufloat pro tuple
def tuple_unc_A(*velicina):
    list_velicin = [*velicina]
    return tuple(uf(np.mean(v), unc_A(v)) for v in list_velicin)

# array do latex
def array_to_latex(*array, dig=1, format='u', style='L', only_return=False):
    output = [f'${u:.{dig}{format}{style}}$' for u in array]
    if only_return:
        return output
    else:
        for s in output:
            print(s)

In [28]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_tenk = pd.read_csv(url)

In [29]:
# tenka vrstva
vln_delka_tenk = 532.1*1e-9
df_tenk_list = ['x1a [mm]', 'x2a [mm]', 'x1b [mm]', 'x2b [mm]', 'x1c [mm]', 'x2c [mm]', 'x1d [mm]', 'x2d [mm]', 'x1e [mm]', 'x2e [mm]', 'x1f [mm]', 'x2f [mm]', 'x1g [mm]', 'x2g [mm]', 'x1h [mm]', 'x2h [mm]']
x1a, x2a, x1b, x2b, x1c, x2c, x1d, x2d, x1e, x2e, x1f, x2f, x1g, x2g, x1h, x2h = df_to_arrays(df_tenk, df_tenk_list, dtype=int)

def tenka(pripona):
    x1 = globals()[f"x1{pripona}"]
    x2 = globals()[f"x2{pripona}"]
    return (x2 / x1) * (vln_delka_tenk / 2)

from string import ascii_lowercase

n=8

stred = np.concatenate([tenka('a'),tenka('b')])
print('Střed:', array_to_latex(ufloat_unc_A(stred)*1e9, only_return=True))

dole = np.concatenate([tenka('c'),tenka('d')])
print('Dole:', array_to_latex(ufloat_unc_A(dole)*1e9, only_return=True))

nahore = np.concatenate([tenka('e'),tenka('f')])
print('Nahoře:', array_to_latex(ufloat_unc_A(nahore)*1e9, only_return=True))

dalsi_stred = np.concatenate([tenka('g'),tenka('h')])
print('Další střed:', array_to_latex(ufloat_unc_A(dalsi_stred)*1e9, only_return=True))

for k in ascii_lowercase[:n]:
    print(k, ufloat_unc_A(tenka(k)))

vysledek = []
for k in ascii_lowercase[:n]:
    vysledek.extend(tenka(k))


tlouska = ufloat_unc_A(vysledek)
array_to_latex(tlouska*1e9, dig=1, format='u')

Střed: ['$101 \\pm 3$']
Dole: ['$87 \\pm 3$']
Nahoře: ['$105 \\pm 3$']
Další střed: ['$104 \\pm 3$']
a (1.05+/-0.04)e-07
b (9.9+/-0.4)e-08
c (8.2+/-0.4)e-08
d (9.11+/-0.22)e-08
e (1.088+/-0.029)e-07
f (1.03+/-0.04)e-07
g (1.06+/-0.04)e-07
h (1.03+/-0.05)e-07
$99 \pm 2$


In [30]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "1193928494"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_vzduch = pd.read_csv(url)

In [31]:
# index lomu vzduchu
d = 0.04
p0 = 99600
deltap = abs((0.005-1.020)*98066.5)
N = np.array(df_vzduch['Počet posunutých proužků N'])
n_vz = 1 + ((N * vln_delka_tenk)/(2*d))*(p0/deltap)
n_vz = ufloat_unc_A(n_vz)
array_to_latex(n_vz)
print(deltap)

n_ref_vz = 1.00026571
jedna = 1
print(((-(n_vz.n - jedna) + (n_ref_vz - jedna)) / (n_ref_vz - jedna))*100)

$1.000253 \pm 0.000004$
99537.49750000001
4.818709294859452


In [32]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "1791918306"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_mrizka = pd.read_csv(url)

In [33]:
# difrakce na mrizce
vln_delka_dif = 632.8*1e-9

df_mrizka_list = ['Vzdálenost mřížka–stínítko xa [mm]', 'y1a vpravo [mm]', 'y1a vlevo [mm]', 'y2a vpravo [mm]', 'y2a vlevo [mm]', 'Vzdálenost mřížka–stínítko xb [mm]', 'y1b vpravo [mm]', 'y1b vlevo [mm]', 'y2b vpravo [mm]', 'y2b vlevo [mm]']
xa, y1a_right, y1a_left, y2a_right, y2a_left, xb, y1b_right, y1b_left, y2b_right, y2b_left = df_to_numpy_matrix(df_mrizka, df_mrizka_list, dtype=int).T
y1a = (y1a_left + y1a_right)/2
y2a = (y2a_left + y2a_right)/2
y1b = (y1b_left + y1b_right)/2
y2b = (y2b_left + y2b_right)/2

def difraction_formule(y, x, m):
    sin_alpha = y/((y**2 + x**2)**0.5)
    d = (m * vln_delka_dif)/sin_alpha
    return d

d_a = np.concatenate([difraction_formule(y1a, xa, 1), difraction_formule(y2a, xa, 2)])
d_a = uf(np.mean(d_a), unc_A(d_a))
array_to_latex(d_a, dig=1, format='u')
array_to_latex(1/d_a/1000, dig=1)
d_b = np.concatenate([difraction_formule(y1b, xb, 1), difraction_formule(y2b, xb, 2)])
d_b = uf(np.mean(d_b), unc_A(d_b))
array_to_latex(d_b, dig=1, format='u')
array_to_latex(1/d_b/1000, dig=1)

$\left(3.380 \pm 0.002\right) \times 10^{-6}$
$295.9 \pm 0.2$
$\left(1.675 \pm 0.001\right) \times 10^{-6}$
$597.1 \pm 0.5$
